# PHƯƠNG ÁN 1 — FASHN VTON v1.5 từng frame · `fashn_perframe`

> **1 notebook = 1 phương án.** File này **chỉ** cài FASHN VTON (Apache-2.0, ~2 GB) — không cài
> diffusers/Wan/sam2, nên không xung đột stack với 2 notebook còn lại.
> Gọi `GET /version` để biết chắc đang chạy bản nào, model nào, case nào.

| | |
|---|---|
| **Model** | [FASHN VTON v1.5](https://github.com/fashn-AI/fashn-vton-1.5) — MMDiT pixel-space, maskless, weights [`fashn-ai/fashn-vton-1.5`](https://huggingface.co/fashn-ai/fashn-vton-1.5) |
| **License** | **Apache-2.0** (dùng thương mại được) |
| **GPU** | **T4 chạy được** (~2 GB weights; T4 không có bf16 → tự hạ fp16) |
| **Đầu vào** | 1 video + **1 ảnh outfit** (flat-lay, nền trong suốt/trắng) |
| **Giữ họa tiết áo** | ✅ đúng ảnh outfit (logo, hoa văn) |
| **Điểm yếu** | có thể **nhấp nháy** giữa các frame; cạnh dài bị model hạ về 864 px rồi phóng lại |

Cơ chế: giải video ra frame → chạy try-on **từng frame** với **cùng một seed** (nên nhiễu giống
nhau giữa các frame, giảm nhấp nháy) → ghép lại bằng ffmpeg đúng fps gốc.

### 4 công tắc quan trọng

- **`paste_back`** (mặc định **bật** từ v3) — model chỉ trả ảnh **486×864** cho khung 720×1280; nếu
  dán cả khung thì phải phóng ngược lại → **mờ toàn bộ, rõ nhất ở nền chi tiết**. Bật `paste_back`
  thì chỉ vùng model thực sự đổi được ghép lên frame **gốc** → nền/mặt sắc nét nguyên bản.
  Rủi ro: viền trang phục đổi rất nhẹ (< ngưỡng 12) có thể bị bỏ sót.
- **`segmentation_free`** (mặc định **tắt** từ v3) — bật thì **không** mask vùng trang phục, model tự
  quyết giữ hay thay lớp áo trong. Áo mới được phình ra ngoài đường viền áo cũ, **nhưng mỗi frame
  nó quyết lại một lần và quyết khác nhau** → đã gặp thật: lớp áo tee biến mất từ giữa clip, rồi
  mọc thêm dây chuyền. Tắt thì mask trước → đầu vào mỗi frame nhất quán; đổi lại áo mới bị bó trong
  đường viền áo cũ.
- **`fast_mode`** — `forward_for_cfg` của FASHN **nhân đôi batch mỗi step** (cond + uncond).
  Bật `fast_mode` thì chỉ chạy 1 lượt forward → **nhanh gấp 2**, toán học tương đương
  `guidance_scale=1.0`. Đánh đổi: mất CFG nên độ trung thực họa tiết có thể giảm.
- **`temporal_smooth`** — trung vị 3 frame, **chỉ ở pixel ít chuyển động** (đo trên frame gốc) nên
  vùng người đang cử động không bị bóng mờ.

## API

| Việc | Endpoint |
|---|---|
| **Đánh dấu bản/model/case đang test** | `GET /version` |
| Đổi outfit cho video (upload file) | `POST /outfit_video` — `multipart`: `video`, `outfit`, + tham số |
| Đổi outfit cho video (URL) | `POST /outfit_video/json` |
| Thử nhanh **1 ảnh** trước khi chạy video | `POST /outfit_image` — `multipart`: `person`, `outfit` |
| Tiến độ | `GET /jobs/{job_id}` → `{status, stage, progress, test_case, filename}` |
| **Tải video về** | `GET /jobs/{job_id}/result` |
| Trạng thái | `GET /health` · Form test tay | `GET /` |
| **Cái gì đang chạy ở đâu** (device/dtype của DiT, provider của DWPose + human-parser, số token) | `GET /selftest` |
| **Đo thật thời gian từng tầng** + suy ra thời gian 105 frame | `POST /benchmark` — `multipart`: `person`, `outfit` |

Tham số: `test_case`, `category` (`tops`=`outfit/upper`, `bottoms`=`outfit/lower`,
`one-pieces`=`outfit/dress`), `garment_photo_type` (`flat-lay`\|`model`), `max_frames`
(`0`=cả video, `24`=test nhanh), `steps` (20/30/50), `guidance`, `seed`,
`paste_back`, `temporal_smooth`, `segmentation_free`, `fast_mode`.

`test_case` đi vào **tên file** và **metadata mp4**:
`<uid>_fashn_perframe_<test_case>_<version>.mp4`, comment
`notebook=... version=... backend=... case=... category=... steps=...`.
Kiểm tra bằng:

```bash
ffprobe -v error -show_entries format_tags=comment -of default=nw=1 output.mp4
```

## Cách chạy

1. Runtime → Change runtime type → **T4 GPU** → **Run all**.
2. Sửa `TEST_CASE` ở cell 2 cho từng lần thử (vd `C1-baseline`, `C2-pasteback`, `C3-smooth`).
3. Đợi dòng `PUBLIC_URL=...`, rồi:

```bash
python test_outfit_video.py --url https://xxx.trycloudflare.com \
  --video ../../input-video/1_input_1980s-highschooler.mp4 \
  --outfit ../../outfit/upper/cloth1.png --category tops \
  --max-frames 24 --test-case C1-baseline
```

## Tốc độ — đo thật trên T4

`steps=30`, `guidance=1.5`: **~75 s/frame** → 105 frame ≈ **2,2 giờ**. Không phải lỗi cấu hình:
attention đã dùng `scaled_dot_product_attention`, DWPose + human-parser đều khởi tạo với
`device=cuda`, DiT load thẳng `.to(cuda, fp16)`. Chậm là do **kiến trúc** — FASHN VTON chạy
**pixel-space** (không có VAE nén), `patch_size=12` trên 864×576 → 3456 token ảnh + 3456 token áo,
28 block, `hidden=1280`. Ước tính ~2,5e13 FLOPs/step; T4 thực đạt ~25 TFLOPS → ~30 s/frame là
**sàn lý thuyết**, ×2 vì CFG.

Các đòn giảm thời gian, theo thứ tự hiệu quả:

| Đòn | Hệ số | 105 frame |
|---|---|---|
| `fast_mode=true` (bỏ CFG) | **2×** | ~1,1 h |
| `steps=20` thay vì 30 | 1,5× | — |
| cả hai | **3×** | **~45 phút** |
| + đổi sang L4 | ~6× | ~22 phút |
| + đổi sang A100 | ~12× | ~11 phút |

Gộp nhiều frame vào cùng batch **không** giúp đáng kể: 6912 token đã làm no GPU ở batch 1.
Chạy `POST /benchmark` để lấy số thật trên đúng GPU bạn đang có.

## Changelog

- **v3** — sửa 2 lỗi đo được ở v2 và thêm 2 đòn tốc độ:
  **(a) nền bị mờ** → `paste_back` mặc định **bật**;
  **(b) lớp áo trong lúc có lúc không** → thêm tham số `segmentation_free`, mặc định **tắt**;
  **(c)** `fast_mode` bỏ CFG → nhanh gấp 2;
  **(d)** thêm `GET /selftest` và `POST /benchmark` để **đo** thay vì phỏng đoán.
- **v2** — sửa cách cài FASHN VTON: `pip install -e` chỉ ghi một file `.pth` vào site-packages, mà
  `.pth` chỉ được đọc lúc Python khởi động → kernel Colab đang chạy báo
  `No module named 'fashn_vton'` (đã gặp thật ở v1). Nay cài thường (copy hẳn vào site-packages) +
  thêm `src/` vào `sys.path` + `import fashn_vton` và kiểm tra weights **ngay trong cell cài đặt**,
  nên lỗi hiện ở cell 2 chứ không đợi tới lúc có request.
- **v1** — bản đầu: try-on từng frame, `paste_back`, `temporal_smooth`, API upload/download,
  `/version`, `/outfit_image`, job store bất đồng bộ có progress, form HTML, cloudflared.

---

## 4 phương án — trạng thái hiện tại

| Notebook | backend | Video-native? | GPU | License | Giữ họa tiết áo |
|---|---|---|---|---|---|
| `OutfitVideo_CatV2TON_Colab.ipynb` | `catv2ton` | **✅ có** | **T4** | phi thương mại + no-derivatives | ✅ |
| `OutfitVideo_FashnPerFrame_Colab.ipynb` | `fashn_perframe` | ❌ từng frame | T4 | Apache-2.0 | ✅ |
| `OutfitVideo_WanAnimate_Colab.ipynb` | `fashn_wan_animate` | ✅ có | A100 40 GB | Apache-2.0 | ✅ |
| `OutfitVideo_LucyEdit_Colab.ipynb` | `lucy_edit` | ✅ có | T4 / L4 | phi thương mại | ❌ theo prompt |

CatV2TON là phương án đầu tiên **vừa video-native, vừa nhận ảnh outfit, vừa chạy được trên T4** —
nên đó là hướng chính hiện nay. FASHN giữ lại làm baseline ảnh đơn và fallback. Wan2.2-Animate
giữ cho lúc có A100 (license thương mại được). Lucy Edit tạm gác: sai paradigm vì chỉ nhận prompt.

Cả 4 notebook dùng **cùng một hợp đồng API**, nên `test_outfit_video.py` chạy được với tất cả —
script tự đọc `GET /version` để biết đang nói chuyện với backend nào.

Kiểm tra notebook sau khi sửa, **không cần GPU / không cần Colab**:

```bash
python smoke_outfit_notebooks.py
```

## 1) Đánh dấu bản / model / case đang test

Sửa `TEST_CASE` mỗi lần thử một cấu hình khác. Giá trị này đi vào **tên file kết quả**, **metadata mp4** và `GET /version` — để sau này không nhầm lẫn giữa các ver / model.

In [ ]:
# ============================================================================
#  OutfitVideo_FashnPerFrame_Colab.ipynb
#  backend = fashn_perframe   (1 notebook = 1 phuong an, KHONG gop nhieu model 1 file)
# ============================================================================
NOTEBOOK_NAME    = 'OutfitVideo_FashnPerFrame_Colab.ipynb'
NOTEBOOK_VERSION = 'v6'        # <-- BUMP moi khi sua notebook + cap nhat CHANGELOG
TEST_CASE        = 'C1-baseline'    #@param {type:"string"}
BACKEND          = 'fashn_perframe'

# Cong bo qua GET /version -> khong bao gio nham ban nao / model nao / case nao
MODELS = [
    {
        "name": "FASHN VTON v1.5",
        "code": "https://github.com/fashn-AI/fashn-vton-1.5",
        "weights": "hf:fashn-ai/fashn-vton-1.5",
        "license": "Apache-2.0",
        "size": "~2 GB"
    }
]
BACKEND_PARAMS = ('outfit', 'category', 'garment_photo_type', 'max_frames', 'steps', 'guidance', 'seed', 'paste_back', 'temporal_smooth', 'segmentation_free', 'fast_mode', 'paste_thr', 'paste_feather')
CHANGELOG = {
    "v1": "try-on tung frame + paste_back + temporal_smooth; API upload/download; /version; /outfit_image",
    "v2": "FIX: pip install -e ghi .pth nen kernel dang chay khong import duoc fashn_vton -> cai thuong + them src vao sys.path + kiem tra import/weights ngay trong cell cai dat",
    "v3": "FIX nen bi mo: paste_back mac dinh TRUE. FIX lop ao trong luc co luc khong: them tham so segmentation_free (mac dinh FALSE). TOC DO: fast_mode bo CFG -> 2x. THEM /selftest va /benchmark de do that thay vi phong doan.",
    "v4": "chuyen paste_back sang cell dung chung (KHONG doi hanh vi) de ca 3 notebook dung chung mot ham",
    "v6": "/jobs/{id} nay tra ve ca error_type + dong code vo + traceback -- truoc do chi co mot dong message nen moi lan job loi la mat mot luot doi log Colab",
    "v5": "FIX THAT cho nen mo: v3 chi phuc hoi 1/3 do net (nen 84->89%, tran encoder 98%) vi mask tinh bang absdiff voi anh DA PHONG TO -- phep phong lam mem canh sac nen mask an ca vao nen (do duoc: phu 37.9% vung bien neon). Nay mask tinh o do phan giai cua model roi moi phong rieng mask. Them tham so paste_thr / paste_feather."
}

import shutil, subprocess


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    except Exception:
        return ''


print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
for m in MODELS:
    print(f"    model: {m['name']}  [{m['license']}]  {m['weights']}")
print('GPU:', _sh(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
      or '!!! CHUA BAT GPU: Runtime > Change runtime type > GPU')
print('GPU yeu cau cho backend nay:', 'T4 (16 GB) tro len')
print('ffmpeg:', shutil.which('ffmpeg') or 'THIEU (cell cai dat se cai)')


## 2) Cài đặt — FASHN VTON v1.5 (Apache-2.0, ~2 GB) + cloudflared

Chỉ cài THÊM lên stack sẵn có của Colab (giữ numpy 2.x / torch của Colab), **không kill kernel** — cùng nguyên tắc với `ImageAI_Colab.ipynb` / `PhotoTools_Colab.ipynb`.

In [ ]:
import importlib
import os, shutil, subprocess, sys

FASHN_REPO    = '/content/fashn-vton-1.5'
FASHN_WEIGHTS = '/content/weights/fashn'

# 1) Web server. Cai 1 lenh duy nhat de pip tu chot 1 bo phien ban nhat quan.
#    Notebook nay CHI dung FASHN VTON -> khong cai diffusers/Wan/sam2, khong xung dot
#    stack voi 2 notebook con lai (moi phuong an 1 file rieng).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'fastapi', 'uvicorn[standard]', 'python-multipart', 'requests'], check=True)

# 2) FASHN VTON v1.5 -- cai --no-deps roi tu them dung cac goi Colab CHUA co.
#    Khong de pip keo lai torch/torchvision/numpy/opencv cua Colab (de vo ABI,
#    da gap kieu loi "cannot import name _center" o cac notebook truoc).
if not os.path.isdir(FASHN_REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/fashn-AI/fashn-vton-1.5.git',
                    FASHN_REPO], check=True)
# KHONG dung `pip install -e`: editable install chi ghi 1 file .pth vao site-packages,
# ma .pth chi duoc `site` doc luc Python KHOI DONG -> kernel Colab dang chay se bao
# "No module named 'fashn_vton'" (da gap o v1). Cai thuong (copy han vao site-packages)
# roi them ca src/ vao sys.path -> import duoc NGAY, khong phai Restart runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', FASHN_REPO, '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'fashn-human-parser>=0.1.1', 'einops', 'safetensors', 'onnxruntime-gpu'], check=False)

_src = os.path.join(FASHN_REPO, 'src')
if os.path.isdir(_src) and _src not in sys.path:
    sys.path.append(_src)
importlib.invalidate_caches()
from fashn_vton import TryOnPipeline   # noqa: F401  -- fail SOM tai day, khong doi den luc co request
import fashn_vton
print('fashn_vton OK ->', fashn_vton.__file__)

import onnxruntime as ort
print('onnxruntime =', ort.__version__, '| providers =', ort.get_available_providers())
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    print('   [canh bao] onnxruntime khong thay GPU -> DWPose/human-parser chay CPU')
    print('   (van chay duoc, cham hon ~1s/frame). Thu: pip install "onnxruntime-gpu==1.20.2"')

# 3) Weights FASHN (~2 GB): model.safetensors + dwpose/*.onnx
if not os.path.exists(os.path.join(FASHN_WEIGHTS, 'model.safetensors')):
    subprocess.run([sys.executable, os.path.join(FASHN_REPO, 'scripts', 'download_weights.py'),
                    '--weights-dir', FASHN_WEIGHTS], check=True, cwd=FASHN_REPO)
for f in ['model.safetensors', 'dwpose/yolox_l.onnx', 'dwpose/dw-ll_ucoco_384.onnx']:
    p = os.path.join(FASHN_WEIGHTS, f)
    print(f'   {f}: {os.path.getsize(p) / 1e6:.0f} MB' if os.path.exists(p) else f'   {f}: THIEU')

# 4) ffmpeg + cloudflared
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=False)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('ffmpeg:', shutil.which('ffmpeg'))
print('cloudflared:', subprocess.run(['/usr/local/bin/cloudflared', '--version'],
                                     capture_output=True, text=True).stdout.strip())

# 7) Kiem tra cuoi: import + weights day du -> neu thieu thi bao NGAY tai cell nay
_missing = [f for f in ('model.safetensors', 'dwpose/yolox_l.onnx', 'dwpose/dw-ll_ucoco_384.onnx')
            if not os.path.exists(os.path.join(FASHN_WEIGHTS, f))]
if _missing:
    raise RuntimeError(f'Thieu weights FASHN: {_missing} -> chay lai cell nay')

import torch
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print(f'=== [{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] cai dat xong ===')

## 3) Tiện ích video (ffmpeg) + nạp model

In [ ]:
import glob, json, os, re, shutil, subprocess, time
import cv2
import numpy as np
import torch
from PIL import Image

JOB_DIR = '/content/jobs'
RES_DIR = '/content/results'
WRK_DIR = '/content/work'
for d in (JOB_DIR, RES_DIR, WRK_DIR):
    os.makedirs(d, exist_ok=True)


def _slug(s, default='case'):
    s = re.sub(r'[^A-Za-z0-9._-]+', '-', (s or '').strip()).strip('-.')
    return (s or default)[:48]


# ---------------------------------------------------------------- ffmpeg/ffprobe
def probe_video(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
         'stream=width,height,r_frame_rate,nb_frames', '-of', 'json', path],
        capture_output=True, text=True, check=True).stdout
    st = json.loads(out)['streams'][0]
    num, den = (st['r_frame_rate'].split('/') + ['1'])[:2]
    fps = float(num) / float(den or 1)
    return int(st['width']), int(st['height']), fps, int(st.get('nb_frames') or 0)


def has_audio(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'a', '-show_entries',
                          'stream=index', '-of', 'csv=p=0', path],
                         capture_output=True, text=True).stdout.strip()
    return bool(out)


def extract_frames(video, out_dir, max_frames=0):
    """Giai video ra PNG o DUNG do phan giai goc (khong resize)."""
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    cmd = ['ffmpeg', '-v', 'error', '-y', '-i', video]
    if max_frames and max_frames > 0:
        cmd += ['-frames:v', str(int(max_frames))]
    cmd += ['-start_number', '0', os.path.join(out_dir, '%06d.png')]
    subprocess.run(cmd, check=True)
    return sorted(glob.glob(os.path.join(out_dir, '*.png')))


def encode_video(frames_dir, fps, out_path, tag='', src_video=None):
    """Ghep PNG -> mp4 h264. Metadata comment mang DU notebook + version + backend + case
    -> mo file mp4 bang ffprobe/exiftool la biet ngay ban nao sinh ra no."""
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    cmd = ['ffmpeg', '-v', 'error', '-y',
           '-framerate', f'{fps:.6f}', '-start_number', '0',
           '-i', os.path.join(frames_dir, '%06d.png')]
    if src_video and has_audio(src_video):
        cmd += ['-i', src_video, '-map', '0:v:0', '-map', '1:a:0', '-c:a', 'aac', '-shortest']
    cmd += ['-c:v', 'libx264', '-preset', 'medium', '-crf', '18', '-pix_fmt', 'yuv420p',
            '-metadata', f'comment={comment}', out_path]
    subprocess.run(cmd, check=True)
    return out_path


def save_frames(frames, out_dir, size=None):
    """frames: list PIL.Image -> PNG 000000.png... (tuy chon resize ve size=(w,h))."""
    os.makedirs(out_dir, exist_ok=True)
    for i, im in enumerate(frames):
        if size and im.size != tuple(size):
            im = im.resize(tuple(size), Image.LANCZOS)
        im.save(os.path.join(out_dir, f'{i:06d}.png'))
    return out_dir


def paste_back_region(orig_bgr, gen_native_bgr, thr=12, feather=9):
    """Ghep vung model thuc su doi len frame GOC, phan con lai giu pixel goc.

    `gen_native_bgr` phai la output o DO PHAN GIAI GOC CUA MODEL (vd 486x864), KHONG
    phai ban da phong to. Mask duoc tinh o do phan giai do roi moi phong rieng mask len.

    Vi sao quan trong -- da do that: neu tinh mask bang absdiff giua anh goc 720x1280 va
    anh model da phong to, thi chinh phep phong lam mem cac canh sac nhat (bien neon) nen
    absdiff vuot nguong ngay tai nen; mask cu phu 37.9% vung bien neon voi alpha TB 0.24,
    keo do net nen chi ve 89% thay vi 98%. Tinh mask o do phan giai model thi khong co
    phep phong nao de gay nham lan.
    """
    H, W = orig_bgr.shape[:2]
    gh, gw = gen_native_bgr.shape[:2]
    small = cv2.resize(orig_bgr, (gw, gh), interpolation=cv2.INTER_AREA)
    d = cv2.absdiff(small, gen_native_bgr).max(axis=2)
    m = (d > thr).astype(np.uint8) * 255
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((11, 11), np.uint8))
    m = cv2.dilate(m, np.ones((5, 5), np.uint8))
    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR)
    k = 2 * int(feather) + 1
    a = (cv2.GaussianBlur(m, (k, k), 0).astype(np.float32) / 255.0)[..., None]
    big = cv2.resize(gen_native_bgr, (W, H), interpolation=cv2.INTER_LANCZOS4)
    return (big.astype(np.float32) * a + orig_bgr.astype(np.float32) * (1 - a)).astype(np.uint8)


def paste_back_pil(orig_pil, gen_pil, thr=12, feather=9):
    """Ban PIL: gen_pil giu nguyen do phan giai goc cua model, KHONG resize truoc khi goi."""
    o = cv2.cvtColor(np.array(orig_pil.convert('RGB')), cv2.COLOR_RGB2BGR)
    g = cv2.cvtColor(np.array(gen_pil.convert('RGB')), cv2.COLOR_RGB2BGR)
    out = paste_back_region(o, g, thr=thr, feather=feather)
    return Image.fromarray(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))


# ---------------------------------------------------------------- anh outfit
def load_garment(path):
    """Anh outfit trong repo la PNG RGBA nen TRONG SUOT (outfit/{upper,lower,dress}/*.png).
    .convert('RGB') se bien nen thanh DEN -> sai voi flat-lay. Phai flatten len nen TRANG."""
    im = Image.open(path)
    if im.mode in ('RGBA', 'LA') or (im.mode == 'P' and 'transparency' in im.info):
        im = im.convert('RGBA')
        bg = Image.new('RGBA', im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return im.convert('RGB')


print(f'[{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] video utils OK')


# ---------------------------------------------------------------- FASHN VTON v1.5
# 'auto' -> bf16 tren Ampere/Ada, fp16 tren T4 (T4 sm75 KHONG co bf16; torch.cuda.
# is_bf16_supported() van tra True nen pipeline tu chon bf16 -> bi emulate va cham).
FASHN_DTYPE = 'auto'          # auto | fp32 | fp16 | bf16
CATEGORIES = ('tops', 'bottoms', 'one-pieces')
_FASHN = None


def get_fashn():
    global _FASHN
    if _FASHN is not None:
        return _FASHN
    from fashn_vton import TryOnPipeline
    t0 = time.time()
    p = TryOnPipeline(weights_dir=FASHN_WEIGHTS)

    want = FASHN_DTYPE
    if want == 'auto':
        cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
        want = 'bf16' if cap[0] >= 8 else ('fp16' if cap[0] >= 7 else 'fp32')
    dt = {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}[want]
    if p.inference_dtype != dt:
        try:
            p.tryon_model.to(dt)
            p.inference_dtype = dt
            print(f'   [fashn] ep dtype -> {want}')
        except Exception as e:
            print(f'   [fashn] khong ep duoc dtype {want}: {e} -> giu nguyen')
    m = p.tryon_model
    dev = next(m.parameters()).device
    print(f'   [fashn] input_shape={m.input_shape} patch={m.patch_size} dtype={p.inference_dtype} '
          f'device={dev} ({time.time() - t0:.0f}s)')
    _FASHN = p
    return _FASHN


def _try(fn, default=None):
    try:
        return fn()
    except Exception:
        return default


def fashn_runtime_info():
    """Bao cao thuc te cai gi dang chay o dau -- de khong phai doan."""
    p = get_fashn()
    m = p.tryon_model
    h, w = m.input_shape
    tok = (h // m.patch_size) * (w // m.patch_size)
    info = {
        'torch_device': str(next(m.parameters()).device),
        'torch_dtype': str(p.inference_dtype),
        'input_shape': [h, w],
        'patch_size': m.patch_size,
        'tokens_per_stream': tok,
        'tokens_total_attn': tok * 2,
        'hidden_size': _try(lambda: m.x_embedder.proj.out_channels),
        'params_M': round(sum(q.numel() for q in m.parameters()) / 1e6, 1),
        'cuda': torch.cuda.is_available(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    try:
        import onnxruntime as ort
        info['onnxruntime_providers'] = ort.get_available_providers()
    except Exception as e:
        info['onnxruntime_providers'] = f'loi: {e}'
    for name, obj in (('dwpose', p.pose_model), ('human_parser', p.hp_model)):
        prov = None
        for attr in ('session', 'ort_session', 'sess'):
            s = getattr(obj, attr, None)
            if s is not None and hasattr(s, 'get_providers'):
                prov = s.get_providers()
                break
        if prov is None:   # DWPose bo 2 session (det + pose)
            subs = [getattr(obj, a, None) for a in
                    ('session_det', 'session_pose', 'det_session', 'pose_session',
                     'pose_estimation', 'detector')]
            provs = []
            for s in subs:
                for attr in ('session', 'ort_session', 'sess'):
                    ss = getattr(s, attr, None) if s is not None else None
                    if ss is not None and hasattr(ss, 'get_providers'):
                        provs.append(ss.get_providers())
            prov = provs or 'khong doc duoc (xem onnxruntime_providers)'
        info[f'{name}_providers'] = prov
        info[f'{name}_device_attr'] = str(getattr(obj, 'device', '?'))
    return info


def _patch_skip_cfg(model):
    """forward_for_cfg NHAN DOI batch moi step (cond + uncond) -> 2x chi phi.
    Tra ve v_u == v_c thi v_guided = v_u + g*(v_c - v_u) = v_c voi MOI guidance,
    tuc dung y nghia guidance_scale=1.0 nhung chi 1 luot forward -> nhanh gap 2.
    Tra lai ham goc de co the bat/tat theo tung request."""
    orig = model.forward_for_cfg

    def fast(*args, **kwargs):
        kw = {k: v for k, v in kwargs.items() if v is not None}
        out = model.forward(*args, **kw)['x']      # mask=None -> toan bo la conditional
        return {'v_c': out, 'v_u': out}

    model.forward_for_cfg = fast
    return orig


def tryon_image(person_pil, garment_pil, category='tops', garment_photo_type='flat-lay',
                steps=30, guidance=1.5, seed=42, segmentation_free=True, fast_mode=False):
    """1 anh nguoi + 1 anh outfit -> 1 anh PIL da mac.

    segmentation_free=True  : KHONG mask vung trang phuc, model tu quyet giu/thay lop ao trong
                              -> ao moi duoc phinh ra ngoai duong vien ao cu, NHUNG moi frame
                              "quyet" lai mot lan va quyet khac nhau (da gap: lop ao trong bien
                              mat giua clip).
    segmentation_free=False : mask vung trang phuc truoc -> dau vao moi frame nhat quan hon,
                              doi lai ao moi bi bo trong duong vien ao cu.
    fast_mode=True          : bo CFG -> nhanh gap 2, tuong duong guidance=1.0.

    Ti le khung hinh duoc giu (pipeline unpad o buoc cuoi) nhung canh dai bi ha ve 864px."""
    if category not in CATEGORIES:
        raise ValueError(f'category phai thuoc {CATEGORIES}, nhan "{category}"')
    p = get_fashn()
    restore = _patch_skip_cfg(p.tryon_model) if fast_mode else None
    try:
        out = p(
            person_image=person_pil, garment_image=garment_pil,
            category=category, garment_photo_type=garment_photo_type,
            num_samples=1, num_timesteps=int(steps), guidance_scale=float(guidance),
            seed=int(seed), segmentation_free=bool(segmentation_free),
        )
    finally:
        if restore is not None:
            p.tryon_model.forward_for_cfg = restore
    return out.images[0]


def fashn_benchmark(person_pil, garment_pil, category='tops', steps=30, seed=42):
    """Do tach bach: DWPose / human-parser / phan dien (diffusion), ca 2 che do.
    Dung de tra loi 'cai gi dang an thoi gian' bang so do, khong phai phong doan."""
    p = get_fashn()
    marks = {'pose': 0.0, 'parse': 0.0}

    # LUU Y: gan __call__ len INSTANCE khong co tac dung -- Python tra cuu dunder tren CLASS.
    pose_cls, pose_orig = type(p.pose_model), type(p.pose_model).__call__
    parse_orig = p.hp_model.predict

    def timed_pose(self, *a, **k):
        t = time.time()
        try:
            return pose_orig(self, *a, **k)
        finally:
            marks['pose'] += time.time() - t

    def timed_parse(*a, **k):
        t = time.time()
        try:
            return parse_orig(*a, **k)
        finally:
            marks['parse'] += time.time() - t

    res = {'steps': steps}
    try:
        pose_cls.__call__ = timed_pose
        p.hp_model.predict = timed_parse
        for label, fast in (('binh_thuong', False), ('fast_mode', True)):
            marks['pose'] = marks['parse'] = 0.0
            t0 = time.time()
            tryon_image(person_pil, garment_pil, category=category, steps=steps,
                        seed=seed, fast_mode=fast)
            total = time.time() - t0
            diff = total - marks['pose'] - marks['parse']
            res[label] = {'tong_s': round(total, 2),
                          'dwpose_s': round(marks['pose'], 2),
                          'human_parser_s': round(marks['parse'], 2),
                          'diffusion_s': round(diff, 2),
                          's_moi_step': round(diff / max(1, steps), 3)}
    finally:
        pose_cls.__call__ = pose_orig
        p.hp_model.predict = parse_orig

    n, f = res['binh_thuong']['tong_s'], res['fast_mode']['tong_s']
    res['fast_mode_nhanh_hon'] = f'{n / f:.2f}x' if f else '?'
    res['uoc_tinh_105_frame'] = {
        'binh_thuong_phut': round(n * 105 / 60, 1),
        'fast_mode_phut': round(f * 105 / 60, 1),
    }
    return res


print(f'[{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] FASHN VTON loader OK')

## 4) Backend `fashn_perframe` — try-on từng frame (+ `paste_back`, `temporal_smooth`)

In [ ]:
def _temporal_median(gen_dir, orig_paths, motion_thr=8):
    """Trung vi 3 frame, CHI ap o pixel it chuyen dong (do bang frame GOC).
    Vung dong giu nguyen frame cua no -> khong bi bong mo khi nguoi chuyen dong."""
    gen_paths = sorted(glob.glob(os.path.join(gen_dir, '*.png')))
    n = len(gen_paths)
    if n < 3:
        return 0
    prev_g, cur_g = cv2.imread(gen_paths[0]), cv2.imread(gen_paths[1])
    prev_o, cur_o = cv2.imread(orig_paths[0]), cv2.imread(orig_paths[1])
    fixed = 0
    for i in range(1, n - 1):
        next_g, next_o = cv2.imread(gen_paths[i + 1]), cv2.imread(orig_paths[i + 1])
        mo = np.maximum(cv2.absdiff(cur_o, prev_o).max(axis=2),
                        cv2.absdiff(cur_o, next_o).max(axis=2))
        static = (mo < motion_thr)[..., None]
        med = np.median(np.stack([prev_g, cur_g, next_g]), axis=0).astype(np.uint8)
        cv2.imwrite(gen_paths[i], np.where(static, med, cur_g))
        fixed += 1
        prev_g, cur_g = cur_g, next_g
        prev_o, cur_o = cur_o, next_o
    return fixed


def run_backend(video, outfit, out_path, test_case='', category='tops',
                garment_photo_type='flat-lay', steps=30, guidance=1.5, seed=42,
                max_frames=0, paste_back=True, temporal_smooth=False,
                segmentation_free=False, fast_mode=False,
                paste_thr=12, paste_feather=9, progress=None):
    """FASHN VTON v1.5 chay TUNG FRAME. T4 chay duoc, Apache-2.0, giu dung hoa van ao.

    paste_back=True (mac dinh tu v3): model tra anh 486x864 cho khung 720x1280, neu dan ca
    khung thi PHONG NGUOC lai -> mo toan bo, ro nhat o nen chi tiet. paste_back chi ghep
    vung doi tro lai frame GOC -> nen/mat sac net nguyen ban.
    segmentation_free=False (mac dinh tu v3): xem docstring tryon_image -- chong hien tuong
    lop ao trong luc co luc khong giua clip.
    fast_mode=True: bo CFG -> nhanh gap 2, tuong duong guidance=1.0."""
    uid = os.path.basename(out_path).split('_')[0]
    src_dir = os.path.join(WRK_DIR, uid, 'src')
    dst_dir = os.path.join(WRK_DIR, uid, 'out')
    os.makedirs(dst_dir, exist_ok=True)

    W, H, fps, _ = probe_video(video)
    frames = extract_frames(video, src_dir, max_frames)
    total = len(frames)
    garment = load_garment(outfit)
    stamp = f'{NOTEBOOK_VERSION}/{BACKEND}/{_slug(test_case)}'
    print(f'[{stamp}] {total} frame {W}x{H} @ {fps:.3f}fps | category={category} '
          f'steps={steps} guidance={guidance} seed={seed} paste_back={paste_back} '
          f'smooth={temporal_smooth} seg_free={segmentation_free} fast={fast_mode}')

    get_fashn()
    t0 = time.time()
    for i, fp in enumerate(frames):
        gen = tryon_image(Image.open(fp).convert('RGB'), garment, category,
                          garment_photo_type, steps, guidance, seed,
                          segmentation_free=segmentation_free, fast_mode=fast_mode)
        if paste_back:
            # Dua anh o DO PHAN GIAI GOC cua model (486x864) vao paste_back -- KHONG phong
            # truoc, vi phep phong chinh la thu lam mask an vao nen (xem docstring).
            gen_bgr = paste_back_region(cv2.imread(fp),
                                        cv2.cvtColor(np.array(gen), cv2.COLOR_RGB2BGR),
                                        thr=paste_thr, feather=paste_feather)
        else:
            gen_bgr = cv2.cvtColor(np.array(gen.resize((W, H), Image.LANCZOS)), cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(dst_dir, f'{i:06d}.png'), gen_bgr)
        if progress:
            progress('tryon', i + 1, total)
        if i == 0 or (i + 1) % 10 == 0 or i + 1 == total:
            el = time.time() - t0
            print(f'   [{stamp}] frame {i + 1}/{total} ({el / (i + 1):.1f}s/frame, '
                  f'con ~{el / (i + 1) * (total - i - 1) / 60:.1f} phut)')

    if temporal_smooth:
        if progress:
            progress('smooth', 0, total)
        print(f'   [{stamp}] temporal_smooth: da loc {_temporal_median(dst_dir, frames)} frame')

    if progress:
        progress('encode', total, total)
    encode_video(dst_dir, fps, out_path, src_video=video,
                 tag=(f'case={_slug(test_case)} category={category} steps={steps} '
                      f'guidance={guidance} seed={seed} paste_back={int(paste_back)} '
                      f'smooth={int(temporal_smooth)} seg_free={int(segmentation_free)} '
                      f'fast={int(fast_mode)} thr={paste_thr} feather={paste_feather} '
                      f'frames={total}'))
    print(f'[{stamp}] XONG sau {(time.time() - t0) / 60:.1f} phut -> {out_path}')
    shutil.rmtree(os.path.join(WRK_DIR, uid), ignore_errors=True)
    return out_path


PRELOAD = [('FASHN VTON v1.5', get_fashn)]
print(f'[{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] backend {BACKEND} OK')

## 5) Chạy server + mở tunnel

Giữ cell này chạy. Đợi dòng `PUBLIC_URL=...` rồi gọi API vào URL đó (bắt đầu bằng `GET /version`).

In [ ]:
import os, re, socket, subprocess, threading, time, traceback, uuid
from typing import Optional
from urllib.parse import urlparse, urlsplit

import requests
import uvicorn
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import FileResponse, HTMLResponse
from pydantic import BaseModel

PORT = 8000
STATE = {'models_ready': False}
app = FastAPI(title=f'Outfit Video Server [{BACKEND}] {NOTEBOOK_VERSION}')


# ============================================================ job store (async)
# Tra job_id ngay roi render o thread nen -> tranh Cloudflare 524 khi job chay vai chuc phut.
JOBS, JOBS_LOCK, RUN_LOCK, JOB_TTL = {}, threading.Lock(), threading.Lock(), 6 * 3600


def _prune_locked():
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v.get('created', now) > JOB_TTL]:
        v = JOBS.pop(k, None)
        try:
            if v and v.get('result') and os.path.exists(v['result']):
                os.remove(v['result'])
        except Exception:
            pass


def _new_job(jid=None):
    jid = jid or uuid.uuid4().hex
    with JOBS_LOCK:
        _prune_locked()
        JOBS[jid] = {'status': 'processing', 'stage': 'queued', 'progress': 0.0,
                     'result': None, 'filename': None, 'error': None, 'created': time.time()}
    return jid


def _progress_cb(jid):
    def cb(stage, done, total):
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['stage'] = stage
                JOBS[jid]['progress'] = round(done / max(1, total), 4)
    return cb


def _run_async(jid, tag, work):
    try:
        with RUN_LOCK:                      # serialize GPU: chi 1 job render 1 luc
            path, filename = work()
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='done', stage='done', progress=1.0,
                                 result=path, filename=filename)
        print(f'[{tag}] xong job {jid[:8]} -> {path}')
    except Exception as e:
        traceback.print_exc()
        # Dua ca TRACEBACK vao job: client goi /jobs/{id} la du chan doan, khong phai
        # doi nguoi dung dan log Colab ve (da mat nhieu luot vi thieu cho nay).
        tb = traceback.format_exc()
        frames = [ln for ln in tb.splitlines() if ln.strip().startswith('File "')]
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='error', stage='error', error=str(e)[:800],
                                 error_type=type(e).__name__,
                                 where=frames[-1].strip()[:300] if frames else None,
                                 traceback=tb[-2500:])
        print(f'[{tag}] LOI job {jid[:8]}: {type(e).__name__}: {e}')


def _spawn(jid, tag, work, test_case=None):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid]['test_case'] = _slug(test_case or TEST_CASE)
    print(f'[{tag}] nhan job {jid[:8]} case={_slug(test_case or TEST_CASE)} '
          f'({NOTEBOOK_NAME} {NOTEBOOK_VERSION}) -> chay nen.')
    threading.Thread(target=_run_async, args=(jid, tag, work), daemon=True).start()
    return {'job_id': jid, 'status': 'processing', 'backend': BACKEND,
            'test_case': _slug(test_case or TEST_CASE),
            'result_url': f'/jobs/{jid}/result',
            'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION}


def _active_jobs():
    with JOBS_LOCK:
        return sum(1 for v in JOBS.values() if v.get('status') == 'processing')


# ============================================================ io helpers
def _download(url, dst):
    sp = urlsplit(url)
    headers = {'User-Agent': 'Mozilla/5.0', 'Referer': f'{sp.scheme}://{sp.netloc}/'}
    r = requests.get(url, stream=True, timeout=300, headers=headers)
    r.raise_for_status()
    with open(dst, 'wb') as f:
        for chunk in r.iter_content(1 << 16):
            f.write(chunk)
    return dst


def _ext(name, default):
    e = os.path.splitext(urlparse(name or '').path or (name or ''))[1].lower()
    return e if e else default


async def _save_upload(up: UploadFile, dst):
    with open(dst, 'wb') as f:
        while True:
            chunk = await up.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
    return dst


def _out_paths(uid, test_case, ext='mp4'):
    """Ten file mang DU: backend + case dang test + version notebook
    -> mo folder ket qua la biet ngay video nao do model nao / case nao sinh ra."""
    fn = f'{uid}_{BACKEND}_{_slug(test_case)}_{NOTEBOOK_VERSION}.{ext}'
    return os.path.join(RES_DIR, fn), fn


STARTED_AT = time.time()


# ============================================================ endpoints chung
@app.get('/version')
def version():
    """Danh dau ro: notebook nao, version nao, model nao, case nao dang test.
    Goi endpoint nay TRUOC moi lan test de khong nham lan giua cac ver/model."""
    try:
        import torch as _t
        gpu = _t.cuda.get_device_name(0) if _t.cuda.is_available() else None
    except Exception:
        gpu = None
    return {
        'notebook': NOTEBOOK_NAME,
        'notebook_version': NOTEBOOK_VERSION,
        'backend': BACKEND,
        'test_case': TEST_CASE,
        'models': MODELS,
        'changelog': CHANGELOG,
        'params': list(BACKEND_PARAMS),
        'gpu': gpu,
        'models_ready': STATE['models_ready'],
        'uptime_s': round(time.time() - STARTED_AT, 1),
        'jobs': {'total': len(JOBS), 'active': _active_jobs(),
                 'done': sum(1 for v in JOBS.values() if v.get('status') == 'done'),
                 'error': sum(1 for v in JOBS.values() if v.get('status') == 'error')},
    }


@app.get('/health')
def health():
    return {'ok': True, 'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION,
            'backend': BACKEND, 'test_case': TEST_CASE,
            'models_ready': STATE['models_ready'], 'active_jobs': _active_jobs(),
            'params': list(BACKEND_PARAMS)}


@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        return {'job_id': job_id, 'status': j['status'], 'stage': j['stage'],
                'progress': j['progress'], 'error': j['error'],
                'elapsed': round(time.time() - j['created'], 1),
                'backend': BACKEND, 'notebook_version': NOTEBOOK_VERSION,
                'test_case': j.get('test_case'), 'filename': j.get('filename'),
                'error_type': j.get('error_type'), 'where': j.get('where'),
                'traceback': j.get('traceback')}


@app.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        status, path, fn, err = j['status'], j['result'], j['filename'], j['error']
    if status == 'processing':
        raise HTTPException(status_code=409, detail='job chua xong')
    if status == 'error':
        raise HTTPException(status_code=500, detail=err or 'job error')
    if not path or not os.path.exists(path):
        raise HTTPException(status_code=410, detail='ket qua khong con (da bi don)')
    return FileResponse(path, media_type='video/mp4' if fn.endswith('.mp4') else 'image/png',
                        filename=fn)


FORM_HEAD = """<!doctype html><meta charset=utf-8><title>Outfit __B__ __V__</title>
<style>body{font:14px system-ui;max-width:640px;margin:2rem auto;padding:0 1rem}
label{display:block;margin:.6rem 0 .2rem;font-weight:600}input,select{width:100%;padding:.4rem}
button{margin-top:1rem;padding:.6rem 1.2rem;font-size:15px}#log{white-space:pre-wrap;margin-top:1rem;
background:#f4f4f5;padding:.8rem;border-radius:6px}</style>
<h2>Outfit Video Server &mdash; backend <code>__B__</code> __V__</h2>
<p><code>__N__</code> &middot; <a href=/version>/version</a></p>
<form id=f><label>Video</label><input type=file name=video accept="video/*" required>
<label>test_case (danh dau lan test nay, se vao ten file)</label><input name=test_case value="__C__">
"""
FORM_TAIL = """<button>Chay</button></form><div id=log>...</div>
<script>
const log = m => document.getElementById('log').textContent = m;
document.getElementById('f').onsubmit = async ev => {
  ev.preventDefault();
  const fd = new FormData(ev.target);
  if (fd.get('outfit') && !fd.get('outfit').size) fd.delete('outfit');
  log('dang upload...');
  const r = await fetch('/outfit_video', {method:'POST', body:fd});
  const j = await r.json();
  if (!j.job_id) return log('LOI: ' + JSON.stringify(j));
  const t0 = Date.now();
  const poll = setInterval(async () => {
    const s = await (await fetch('/jobs/' + j.job_id)).json();
    log(`job ${j.job_id}\\nstage=${s.stage} progress=${(s.progress*100).toFixed(1)}% ` +
        `elapsed=${((Date.now()-t0)/1000).toFixed(0)}s\\nstatus=${s.status}` +
        (s.error ? '\\nERROR: ' + s.error : ''));
    if (s.status === 'done') { clearInterval(poll);
      document.getElementById('log').innerHTML +=
        `<br><br><a href="/jobs/${j.job_id}/result" download>TAI VIDEO VE</a>`; }
    if (s.status === 'error') clearInterval(poll);
  }, 3000);
};
</script>"""


@app.get('/', response_class=HTMLResponse)
def form():
    html = FORM_HEAD + FORM_FIELDS + FORM_TAIL
    for _k, _v in (('__B__', BACKEND), ('__V__', NOTEBOOK_VERSION),
                   ('__N__', NOTEBOOK_NAME), ('__C__', TEST_CASE)):
        html = html.replace(_k, _v)
    return html


FORM_FIELDS = """<label>Anh outfit (flat-lay, nen trong suot/trang)</label>
<input type=file name=outfit accept="image/*" required>
<label>category</label><select name=category>
<option>tops</option><option>bottoms</option><option>one-pieces</option></select>
<label>garment_photo_type</label><select name=garment_photo_type>
<option>flat-lay</option><option>model</option></select>
<label>max_frames (0 = ca video, 24 = test nhanh)</label><input name=max_frames type=number value=24>
<label>steps (20 nhanh / 30 can bang / 50 net)</label><input name=steps type=number value=30>
<label>guidance</label><input name=guidance type=number step=0.1 value=1.5>
<label>seed</label><input name=seed type=number value=42>
<label><input type=checkbox name=paste_back value=true checked style=width:auto> paste_back (giu nen goc sac net)</label>
<label><input type=checkbox name=temporal_smooth value=true style=width:auto> temporal_smooth</label>
<label><input type=checkbox name=segmentation_free value=true style=width:auto> segmentation_free (bat = ao phinh tu do nhung lop ao trong hay nhay)</label>
<label><input type=checkbox name=fast_mode value=true style=width:auto> fast_mode (bo CFG, nhanh gap 2, = guidance 1.0)</label>
<label>paste_thr (nguong mask; cao hon = mask chat hon, nen net hon)</label><input name=paste_thr type=number value=12>
<label>paste_feather (do nhoe vien mask)</label><input name=paste_feather type=number value=9>
"""


@app.post('/outfit_video')
async def outfit_video(
    video: Optional[UploadFile] = File(None),
    outfit: Optional[UploadFile] = File(None),
    video_url: Optional[str] = Form(None),
    outfit_url: Optional[str] = Form(None),
    test_case: str = Form(''),
    category: str = Form('tops'),
    garment_photo_type: str = Form('flat-lay'),
    max_frames: int = Form(0),
    steps: int = Form(30),
    guidance: float = Form(1.5),
    seed: int = Form(42),
    paste_back: bool = Form(True),
    temporal_smooth: bool = Form(False),
    segmentation_free: bool = Form(False),
    fast_mode: bool = Form(False),
    paste_thr: int = Form(12),
    paste_feather: int = Form(9),
    job_id: Optional[str] = Form(None),
):
    """Upload video + anh outfit -> tra job_id. Tai ket qua o GET /jobs/{id}/result."""
    if not video and not video_url:
        raise HTTPException(status_code=400, detail='can file "video" hoac "video_url"')
    if not outfit and not outfit_url:
        raise HTTPException(status_code=400, detail=f'backend {BACKEND} can file "outfit" hoac "outfit_url"')
    if category not in CATEGORIES:
        raise HTTPException(status_code=400, detail=f'category phai thuoc {list(CATEGORIES)}')

    uid = uuid.uuid4().hex[:8]
    vpath = os.path.join(JOB_DIR, f'{uid}_in{_ext(video.filename if video else video_url, ".mp4")}')
    if video:
        await _save_upload(video, vpath)
    else:
        _download(video_url, vpath)
    opath = os.path.join(JOB_DIR, f'{uid}_outfit{_ext(outfit.filename if outfit else outfit_url, ".png")}')
    if outfit:
        await _save_upload(outfit, opath)
    else:
        _download(outfit_url, opath)

    case = test_case or TEST_CASE
    jid = _new_job(job_id)
    out_path, fn = _out_paths(uid, case)
    prog = _progress_cb(jid)

    def work():
        return run_backend(vpath, opath, out_path, test_case=case, category=category,
                           garment_photo_type=garment_photo_type, steps=steps,
                           guidance=guidance, seed=seed, max_frames=max_frames,
                           paste_back=paste_back, temporal_smooth=temporal_smooth,
                           segmentation_free=segmentation_free, fast_mode=fast_mode,
                           paste_thr=paste_thr, paste_feather=paste_feather,
                           progress=prog), fn

    return _spawn(jid, BACKEND, work, case)


class VideoJob(BaseModel):
    video_url: str
    outfit_url: str
    test_case: str = ''
    category: str = 'tops'
    garment_photo_type: str = 'flat-lay'
    max_frames: int = 0
    steps: int = 30
    guidance: float = 1.5
    seed: int = 42
    paste_back: bool = True
    temporal_smooth: bool = False
    segmentation_free: bool = False
    fast_mode: bool = False
    paste_thr: int = 12
    paste_feather: int = 9
    job_id: Optional[str] = None


@app.post('/outfit_video/json')
def outfit_video_json(job: VideoJob):
    """Ban JSON dung URL -- tuong thich cach goi cua cac notebook khac trong repo."""
    if job.category not in CATEGORIES:
        raise HTTPException(status_code=400, detail=f'category phai thuoc {list(CATEGORIES)}')
    uid = uuid.uuid4().hex[:8]
    case = job.test_case or TEST_CASE
    jid = _new_job(job.job_id)
    out_path, fn = _out_paths(uid, case)
    prog = _progress_cb(jid)

    def work():
        vpath = _download(job.video_url, os.path.join(JOB_DIR, f'{uid}_in{_ext(job.video_url, ".mp4")}'))
        opath = _download(job.outfit_url, os.path.join(JOB_DIR, f'{uid}_outfit{_ext(job.outfit_url, ".png")}'))
        return run_backend(vpath, opath, out_path, test_case=case, category=job.category,
                           garment_photo_type=job.garment_photo_type, steps=job.steps,
                           guidance=job.guidance, seed=job.seed, max_frames=job.max_frames,
                           paste_back=job.paste_back, temporal_smooth=job.temporal_smooth,
                           segmentation_free=job.segmentation_free, fast_mode=job.fast_mode,
                           paste_thr=job.paste_thr, paste_feather=job.paste_feather,
                           progress=prog), fn

    return _spawn(jid, BACKEND, work, case)


@app.post('/outfit_image')
async def outfit_image(
    person: UploadFile = File(...),
    outfit: UploadFile = File(...),
    test_case: str = Form(''),
    category: str = Form('tops'),
    garment_photo_type: str = Form('flat-lay'),
    steps: int = Form(30),
    guidance: float = Form(1.5),
    seed: int = Form(42),
    segmentation_free: bool = Form(False),
    fast_mode: bool = Form(False),
):
    """Thu 1 ANH truoc khi ton vai chuc phut cho video -- kiem tra category/anh outfit co dung."""
    uid = uuid.uuid4().hex[:8]
    ppath = await _save_upload(person, os.path.join(JOB_DIR, f'{uid}_p{_ext(person.filename, ".png")}'))
    opath = await _save_upload(outfit, os.path.join(JOB_DIR, f'{uid}_o{_ext(outfit.filename, ".png")}'))
    case = test_case or TEST_CASE
    out_path, fn = _out_paths(uid, case, ext='png')
    jid = _new_job()

    def work():
        tryon_image(Image.open(ppath).convert('RGB'), load_garment(opath), category,
                    garment_photo_type, steps, guidance, seed,
                    segmentation_free=segmentation_free, fast_mode=fast_mode).save(out_path)
        return out_path, fn

    return _spawn(jid, f'{BACKEND}:image', work, case)


@app.get('/selftest')
def selftest():
    """Cai gi dang chay o dau: device/dtype cua DiT, provider cua DWPose + human-parser,
    so token, so tham so. Khong ton GPU, tra ngay."""
    return fashn_runtime_info()


@app.post('/benchmark')
async def benchmark(
    person: UploadFile = File(...),
    outfit: UploadFile = File(...),
    category: str = Form('tops'),
    steps: int = Form(30),
    seed: int = Form(42),
):
    """Do THAT thoi gian tung tang (DWPose / human-parser / diffusion) o ca 2 che do
    binh thuong va fast_mode, roi suy ra thoi gian cho 105 frame. Chay 2 anh -> vai phut."""
    uid = uuid.uuid4().hex[:8]
    ppath = await _save_upload(person, os.path.join(JOB_DIR, f'{uid}_bp{_ext(person.filename, ".png")}'))
    opath = await _save_upload(outfit, os.path.join(JOB_DIR, f'{uid}_bo{_ext(outfit.filename, ".png")}'))
    jid = _new_job()

    def work():
        res = fashn_benchmark(Image.open(ppath).convert('RGB'), load_garment(opath),
                              category=category, steps=steps, seed=seed)
        res['runtime'] = fashn_runtime_info()
        out = os.path.join(RES_DIR, f'{uid}_benchmark_{NOTEBOOK_VERSION}.json')
        import json as _json
        with open(out, 'w', encoding='utf-8') as f:
            _json.dump(res, f, ensure_ascii=False, indent=2)
        print('[benchmark]', _json.dumps(res, ensure_ascii=False))
        return out, os.path.basename(out)

    return _spawn(jid, f'{BACKEND}:benchmark', work, 'benchmark')


# ============================================================ preload + tunnel
print(f'>>> [{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] backend={BACKEND} case={TEST_CASE}')
print(f'>>> Preload model (lan dau co the vai chuc giay)...')
_ok = True
for _name, _fn in PRELOAD:
    _t0 = time.time()
    try:
        _fn()
        print(f'   [preload] {_name} OK ({time.time() - _t0:.0f}s)')
    except Exception as _e:
        _ok = False
        traceback.print_exc()
        print(f'   [preload] {_name} LOI: {_e} (se lazy-load lai khi co request)')
STATE['models_ready'] = _ok
print('>>> MODEL SAN SANG.' if _ok else '>>> MODEL CHUA SAN SANG (xem log tren).')


def _port_in_use(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', p)) == 0


if _port_in_use(PORT):
    print(f'uvicorn da chay san o port {PORT} (chay lai cell) -> KHONG khoi dong lai.')
else:
    threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT,
                                                log_level='warning'), daemon=True).start()
    time.sleep(3)

try:
    subprocess.run(['pkill', '-f', 'cloudflared tunnel'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
except Exception:
    pass

# Bao PUBLIC_URL len api (giong PhotoTools_Colab): bot bom window.__ai_boot truoc khi mo notebook.
AI_API_BASE = 'https://api.downloadvideo.vn'


def _read_ai_boot(tries=6, wait=5):
    import json as _json
    try:
        from google.colab import output as _out
    except Exception as e:
        print('   [report] khong co google.colab.output:', e)
        return None
    for i in range(1, tries + 1):
        try:
            raw = _out.eval_js('JSON.stringify(window.__ai_boot || null)', timeout_sec=20)
            boot = _json.loads(raw) if raw and raw != 'null' else None
            if boot and boot.get('account') and boot.get('token'):
                return boot
            print(f'   [report] lan {i}/{tries}: chua thay window.__ai_boot')
        except Exception as e:
            print(f'   [report] lan {i}/{tries}: eval_js loi: {e}')
        time.sleep(wait)
    return None


def _report_public_url(url):
    boot = _read_ai_boot()
    if not boot:
        print('   [report] KHONG co __ai_boot (chay tay?) -> bot se quet dong PUBLIC_URL=.')
        return False
    api = (boot.get('api') or AI_API_BASE).rstrip('/')
    body = {'account': boot['account'], 'token': boot['token'], 'url': url}
    hdr = {'ngrok-skip-browser-warning': '1', 'User-Agent': 'colab-tunnel-report/1.0'}
    for attempt in range(1, 25):
        try:
            r = requests.post(f'{api}/api/c/ai/profiles/tunnel', json=body, headers=hdr, timeout=20)
            print(f'   [report] {attempt}/24 -> {r.status_code} {r.text[:200]}')
            if r.status_code == 200 and r.json().get('success'):
                return True
            if r.status_code in (400, 401):
                return False
        except Exception as e:
            print(f'   [report] {attempt}/24 loi mang: {e}')
        time.sleep(15)
    return False


proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n\nPUBLIC_URL=' + public_url + '\n', flush=True)
        break

if public_url:
    threading.Thread(target=_report_public_url, args=(public_url,), daemon=True).start()

print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
print('URL:', public_url)
print('Endpoints: GET /version (danh dau ban/model/case) | GET /health | GET / (form test)')
print('           POST /outfit_video (multipart upload) | POST /outfit_video/json (URL)')
print('           GET /jobs/{id} | GET /jobs/{id}/result (tai video ve)')
print('>>> SAN SANG. Giu cell nay chay.')
while True:
    line = proc.stdout.readline()
    if not line:
        break
    if 'ERR' in line or 'error' in line.lower():
        print(line, end='')